# CPHD → SICD Conversion — Automated Image Formation and NITF Export

**Objective**: Automatically select and apply the optimal image formation algorithm based on CPHD metadata, form a complex SAR image, and export as a SICD NITF file.

## Overview

This notebook demonstrates the **end-to-end CPHD-to-SICD pipeline**:
1. **Metadata-driven algorithm selection**: Choose PFA (spotlight) or RDA (stripmap) based on collection mode
2. **Chip extraction**: Process a 1024×1024 subset for fast demonstration (seconds vs. minutes)
3. **Image formation**: Apply the selected algorithm
4. **SICD metadata construction**: Build SICD XML from CPHD metadata
5. **NITF export**: Write the formed image as a SICD-compliant NITF file

All parameters (waveform, geometry, frequency) are derived **entirely from the CPHD file** — no sensor-specific knowledge required.

## Theory

### Algorithm Selection Rules

| CPHD Radar Mode | Default Algorithm | Rationale |
|-----------------|-------------------|----------|
| `SPOTLIGHT` | PFA | Circular k-space annulus, wide aperture |
| `STRIPMAP` | RDA | Linear aperture, Doppler centroid variation |
| `DYNAMIC STRIPMAP` | RDA | Beam steering, block-wise Doppler estimation |

### SICD Metadata Mapping

CPHD → SICD metadata conversion:
- **CollectionInfo**: Derived from `CPHD/Data/Radar_Mode`, `TX_Frequency`
- **ImageData**: `NumRows`, `NumCols` from formed image shape
- **Grid**: `ImagePlane` (slant/ground), `Row/Col` directions, pixel spacing
- **ImageFormation**: `ImageFormAlgo` (PFA, RGAZCOMP), `RcvChanProc/Identifier`
- **SCPCOA**: Scene center point (SCP) and center of aperture (COA)
- **RMA/RMAT**: Polarimetric calibration matrices (if multi-pol)

### NITF Structure

Output SICD NITF contains:
- **Image segment**: Complex I/Q data (magnitude/phase or real/imag)
- **SICD DES**: XML metadata as Data Extension Segment
- **NITF headers**: File-level and image-level security/classification

### Modules Used

| Module | Purpose |
|--------|--------|
| `grdl.IO` | `get_reader('cphd', ...)`, `SICDWriter` |
| `grdl.IO.sar.cphd_to_sicd` | `build_sicd_metadata()` |
| `grdl.image_processing.sar` | PFA, RDA, StripmapPFA, FFBP |
| `napari` | Interactive visualization (optional) |

## Preamble — Version Check

In [ ]:
import gc
import grdl
from packaging.version import parse as parse_version

required = "0.6.1"
current = grdl.__version__

if parse_version(current) < parse_version(required):
    raise RuntimeError(
        f"GRDL {required}+ required. Found {current}. "
        f"Update: pip install --upgrade grdl"
    )

print(f"✓ GRDL {current}")

## Imports

In [ ]:
from pathlib import Path

import numpy as np

from grdl.IO import get_reader
from grdl.IO.sar import SICDWriter
from grdl.IO.sar.cphd_to_sicd import build_sicd_metadata
from grdl.image_processing.sar.image_formation import (
    PolarFormatAlgorithm,
    RangeDopplerAlgorithm,
    CollectionGeometry,
    PolarGrid,
)

### Metadata Inspection 1: Collection Info & Radar Mode

The `collection_info` block identifies the sensor and collection mode. The **radar_mode** field determines which image formation algorithm to use downstream.

In [ ]:
# Temporary path config for metadata inspection (will be replaced by full config later)
from pathlib import Path
CPHD_PATH_TEMP = Path("/path/to/your/cphd_file.cphd")  # Set this in Configuration section below

# Note: This cell will fail if CPHD_PATH_TEMP is not set. Skip to Configuration section to set the path.
if CPHD_PATH_TEMP.exists():
    with get_reader('cphd', CPHD_PATH_TEMP) as reader:
        meta = reader.metadata
        ci = meta.collection_info
        
        print(f"Collection Info:")
        print(f"  Radar Mode     : {ci.radar_mode if ci else 'N/A'}")
        print(f"  Radar Mode ID  : {ci.radar_mode_id if ci else 'N/A'}")
        print(f"  Collector Name : {ci.collector_name if ci else 'N/A'}")
        print(f"  Classification : {ci.classification if ci else 'N/A'}")
        print(f"\\n  → Radar Mode determines algorithm: SPOTLIGHT → PFA, STRIPMAP → RDA")
else:
    print("⚠ CPHD path not set — configure in next section")

In [ ]:
if CPHD_PATH_TEMP.exists():
    with get_reader('cphd', CPHD_PATH_TEMP) as reader:
        meta = reader.metadata
        gp = meta.global_params
        
        print(f"Global Parameters:")
        print(f"  Center Freq (GHz): {gp.center_frequency / 1e9:.3f}")
        print(f"  Bandwidth (MHz)  : {gp.bandwidth / 1e6:.3f}")
        print(f"  Phase SGN        : {gp.phase_sgn:+d}")
        
        # Compute wavelength and range resolution
        c = 299792458.0  # Speed of light (m/s)
        wavelength = c / gp.center_frequency
        range_res = c / (2 * gp.bandwidth)
        
        print(f"\\nDerived Quantities:")
        print(f"  Wavelength (cm)  : {wavelength * 100:.2f}")
        print(f"  Range Resolution : {range_res:.3f} m")
        
        # Channel dimensions
        channel = meta.data.channels[0]
        print(f"\\nChannel 0:")
        print(f"  Num Vectors (pulses)  : {channel.num_vectors:,}")
        print(f"  Num Samples (per pulse): {channel.num_samples:,}")
        print(f"  Signal shape          : ({channel.num_vectors}, {channel.num_samples}) complex64")
        print(f"  Data volume           : {channel.num_vectors * channel.num_samples * 8 / (1024**3):.2f} GB")
else:
    print("⚠ CPHD path not set")

### Metadata Inspection 2: Global Parameters & Waveform

Global parameters define the signal domain, center frequency, and bandwidth. These directly control range resolution: $\rho_r = c / (2B)$ where $B$ is bandwidth.

## Understanding CPHD Structure (Before Formation)

**Why this section exists**: Before forming an image, it's critical to understand the CPHD file contents. This section teaches you:
- How to inspect collection mode (spotlight vs stripmap) → determines which algorithm to use
- How to extract waveform parameters (bandwidth, center frequency) → defines range resolution
- How to read platform trajectory (PVP arrays) → defines azimuth resolution
- How to detect SRP drift → validates collection mode detection

**What is CPHD?** Compensated Phase History Data — the NGA standard for distributing SAR raw data. It contains:
- **Signal array**: Complex baseband echo samples `(num_vectors, num_samples)`
- **Metadata**: Collection info, waveform params, global params
- **PVP arrays**: Per-pulse navigation (platform position, velocity, time)

The sections below walk through each metadata block.

## Configuration — User Paths

In [ ]:
# Input: CPHD file
CPHD_PATH = Path("/path/to/your/cphd_file.cphd")

# Output: SICD NITF file
OUTPUT_PATH = CPHD_PATH.with_suffix(".nitf")
OUTPUT_PATH = OUTPUT_PATH.parent / OUTPUT_PATH.name.replace("cphd", "sicd")

# Validate input
assert CPHD_PATH.exists(), f"CPHD file not found: {CPHD_PATH}"
assert CPHD_PATH.suffix.lower() in [".cphd", ".cph"], f"Expected CPHD file, got {CPHD_PATH.suffix}"

print(f"✓ Input:  {CPHD_PATH.name}")
print(f"✓ Output: {OUTPUT_PATH.name}")

## Step 1: Load CPHD Metadata and Select Algorithm

In [ ]:
with get_reader('cphd', CPHD_PATH) as reader:
    meta = reader.metadata
    radar_mode = meta.collection_info.radar_mode.upper()
    
    # Algorithm selection rules
    algorithm_map = {
        'SPOTLIGHT': 'PFA',
        'STRIPMAP': 'RDA',
        'DYNAMIC STRIPMAP': 'RDA',
    }
    
    selected_algorithm = algorithm_map.get(radar_mode, 'PFA')  # Default to PFA
    
    print(f"Collection mode: {radar_mode}")
    print(f"Selected algorithm: {selected_algorithm}")
    print(f"Center frequency: {meta.global_params.center_frequency / 1e9:.2f} GHz")
    
    # Extract signal dimensions
    channel = meta.data.channels[0]
    n_vectors = channel.num_vectors
    n_samples = channel.num_samples
    
    print(f"Phase history: {n_vectors} vectors × {n_samples} samples")

## Step 2: Configure Algorithm Parameters

In [ ]:
# PFA parameters (used if SPOTLIGHT)
pfa_params = {
    'grid_mode': 'inscribed',
    'range_oversample': 1.0,
    'azimuth_oversample': 1.0,
    'projection': 'slant',
    'scene_sizing': 'full',
}

# RDA parameters (used if STRIPMAP)
rda_params = {
    'block_size': 'auto',
    'overlap': 0.5,
    'range_weighting': None,
    'azimuth_weighting': None,
}

# Select parameters based on algorithm
if selected_algorithm == 'PFA':
    algo_params = pfa_params
    print("Using PFA parameters:")
else:
    algo_params = rda_params
    print("Using RDA parameters:")

for k, v in algo_params.items():
    print(f"  {k}: {v}")

## Step 3: Configure Chip Extraction

**Chip sizing**: The chip dimensions are in phase history space (vectors × samples), not image pixels. The formed image size may differ due to oversampling and grid mode.

**Processing Strategy**: To reduce computation time, we process only a **chip** (spatial subset) of the phase history instead of the full collect. A 1024×1024 chip typically forms in seconds vs. minutes for a full image. We will need to do later processing of the metadata to make the PVP match the size of our chip.

In [ ]:
# Chip extraction parameters
CHIP_SIZE = 1024  # Phase history dimensions (vectors × samples)

# Determine chip bounds (centered in phase history)
with get_reader('cphd', CPHD_PATH) as reader:
    full_shape = reader.get_shape()
    
    # Center the chip
    row_center = full_shape[0] // 2
    col_center = full_shape[1] // 2
    
    row_start = max(0, row_center - CHIP_SIZE // 2)
    row_end = min(full_shape[0], row_start + CHIP_SIZE)
    col_start = max(0, col_center - CHIP_SIZE // 2)
    col_end = min(full_shape[1], col_start + CHIP_SIZE)
    
    print(f"Full phase history: {full_shape[0]} vectors × {full_shape[1]} samples")
    print(f"Chip extraction: [{row_start}:{row_end}, {col_start}:{col_end}]")
    print(f"Chip size: {row_end - row_start} × {col_end - col_start}")

## Step 4: Form SAR Image from Chip

In [ ]:
import gc

with get_reader('cphd', CPHD_PATH) as reader:
    # Read chip first
    print(f"Reading {CHIP_SIZE}×{CHIP_SIZE} chip from phase history...")
    signal = reader.read_chip(
        row_start=row_start,
        row_end=row_end,
        col_start=col_start,
        col_end=col_end,
    )
    
    # Create geometry from full metadata, then subset PVP to match chip - NOT NECESSARY WHEN PROCESSING FULL IMAGE
    full_geometry = CollectionGeometry(reader.metadata)
    
    # Subset geometry arrays to match the chipped vectors
    # Only slice the per-vector arrays (not scalars like wavelength)
    geometry = CollectionGeometry.__new__(CollectionGeometry)
    geometry._pvp = full_geometry._pvp
    geometry._metadata = full_geometry._metadata
    geometry.c = full_geometry.c
    
    # Copy scalar/constant attributes
    for attr in ['fxss', 'fx0', 'toa1', 'toa2', 'amp_sf']:
        if hasattr(full_geometry, attr):
            val = getattr(full_geometry, attr)
            if val is not None and len(val) == full_geometry.npulses:
                setattr(geometry, attr, val[row_start:row_end])
            else:
                setattr(geometry, attr, val)
    
    # Slice per-vector arrays
    geometry.time = full_geometry.time[row_start:row_end]
    geometry.npulses = len(geometry.time)
    geometry.nsamples = signal.shape[1]
    geometry.fx1 = full_geometry.fx1[row_start:row_end]
    geometry.fx2 = full_geometry.fx2[row_start:row_end]
    
    # Slice geometry vectors (ECEF positions, etc.)
    for attr in ['srp_ecf', 'arp_ecf', 'arp_vel_ecf', 'range_vec', 
                 'graz_ang', 'twist_ang', 'slope_ang', 'k_sf', 'phi']:
        if hasattr(full_geometry, attr):
            val = getattr(full_geometry, attr)
            if val is not None and hasattr(val, '__len__') and len(val) == full_geometry.npulses:
                setattr(geometry, attr, val[row_start:row_end])
            else:
                setattr(geometry, attr, val)
    
    # Scalars that don't need slicing
    for attr in ['graz_ang_coa', 'r0', 'r_ca', 'srp_coa_ecf', 'range_uvect_ecf', 'az_uvect_ecf',
                 'arp_poly_x', 'arp_poly_y', 'arp_poly_z']:
        if hasattr(full_geometry, attr):
            setattr(geometry, attr, getattr(full_geometry, attr))
    
    # Instantiate processor with the chipped geometry
    if selected_algorithm == 'PFA':
        grid = PolarGrid(
            geometry=geometry,
            grid_mode=pfa_params['grid_mode'],
            range_oversample=pfa_params['range_oversample'],
            azimuth_oversample=pfa_params['azimuth_oversample'],
            scene_sizing=pfa_params['scene_sizing'],
        )
        processor = PolarFormatAlgorithm(grid=grid)
    else:
        processor = RangeDopplerAlgorithm(
            metadata=reader.metadata,
            range_weighting=rda_params['range_weighting'],
            azimuth_weighting=rda_params['azimuth_weighting'],
            block_size=rda_params['block_size'],
            overlap=rda_params['overlap'],
        )
    
    # Form image
    print(f"Running {selected_algorithm} image formation on chip...")
    image = processor.form_image(signal, geometry)

# Free CPHD signal (no longer needed)
del signal, geometry, full_geometry
gc.collect()

# Compute statistics
magnitude = np.abs(image)
print(f"✓ Image formed: {image.shape}, dtype: {image.dtype}")
print(f"  Magnitude range: [{magnitude.min():.2e}, {magnitude.max():.2e}]")
print(f"  Mean magnitude: {magnitude.mean():.2e}")

## Step 5: Build SICD Metadata from CPHD

In [ ]:
with get_reader('cphd', CPHD_PATH) as reader:
    # Rebuild geometry for metadata construction (need full metadata, not chipped)
    full_geometry = CollectionGeometry(reader.metadata)
    
    # Map algorithm to SICD ImageFormAlgo enum
    # RDA produces RGAZCOMP output (range-azimuth compression)
    sicd_algo_map = {'PFA': 'PFA', 'RDA': 'RGAZCOMP'}
    sicd_algo = sicd_algo_map[selected_algorithm]
    
    # Build SICD metadata from CPHD metadata and algorithm type
    sicd_meta = build_sicd_metadata(
        cphd_meta=reader.metadata,
        geometry=full_geometry,
        image_shape=image.shape,
        image_form_algo=sicd_algo,
    )

print("✓ SICD metadata constructed")
print(f"  ImageFormAlgo: {sicd_meta.image_formation.image_form_algo}")
print(f"  Grid type: {sicd_meta.grid.image_plane}")
print(f"  SCP: ({sicd_meta.geo_data.scp.llh.lat:.4f}, {sicd_meta.geo_data.scp.llh.lon:.4f})")

## Step 6: Write SICD NITF File

In [ ]:
# Write complex image as SICD NITF
with SICDWriter(OUTPUT_PATH, metadata=sicd_meta) as writer:
    writer.write(data=image)

print(f"✓ SICD NITF written: {OUTPUT_PATH}")
print(f"  File size: {OUTPUT_PATH.stat().st_size / 1024 / 1024:.1f} MB")

# Free complex image (already written to disk)
del image
gc.collect()

## Step 7: Verify Output — Read Back SICD

In [ ]:
# Read the formed SICD file
with get_reader('sicd', OUTPUT_PATH) as reader:
    verify_meta = reader.metadata
    verify_shape = reader.get_shape()
    
    print("✓ SICD verification:")
    print(f"  Image shape: {verify_shape}")
    print(f"  Pixel type: {verify_meta.image_data.pixel_type}")
    print(f"  Collection start: {verify_meta.timeline.collect_start}")
    print(f"  Algorithm: {verify_meta.image_formation.image_form_algo}")

## Step 8 (Optional): Visualize Formed Image

In [ ]:
import napari

# dB-scale magnitude
magnitude_db = 20 * np.log10(magnitude + 1e-12)
vmax = magnitude_db.max()
vmin = vmax - 50.0

viewer = napari.Viewer(title=f"CPHD Chip → SICD ({selected_algorithm})")
viewer.add_image(
    magnitude_db,
    name=f"{selected_algorithm} Image (dB)",
    colormap="gray",
    contrast_limits=[vmin, vmax],
)

# Simplify viewer UI - hide unnecessary controls
viewer.window._qt_viewer.dockLayerControls.setVisible(False)  # Hide layer controls dock
viewer.window._qt_viewer.dockConsole.setVisible(False)  # Hide console
try:
    viewer.window._qt_viewer.activityDock.setVisible(False)  # Hide activity dock if present
except AttributeError:
    pass

print("✓ Napari viewer opened")
print(f"  Algorithm: {selected_algorithm}")
print(f"  Chip size: {CHIP_SIZE}×{CHIP_SIZE}")
print(f"  Output: {OUTPUT_PATH.name}")
print("\nExecution paused — close the napari window to continue and free memory...")
napari.run()

print("\n✓ Viewer closed — cleaning up memory...")
del magnitude, magnitude_db, vmax, vmin, viewer
gc.collect()
print("✓ Memory cleanup complete")

## Physical Interpretation

### Chip-Based Processing Strategy

This notebook uses **chip extraction** for fast demonstration:
- Extracts a centered 1024×1024 subset of phase history
- Forms image in seconds instead of minutes
- Full-collect processing follows identical workflow, just call `read_full()` instead of `read_chip()`

### Metadata-Driven Workflow

This notebook demonstrates **zero-configuration** image formation:
- No hardcoded sensor parameters
- No manual algorithm selection
- All information extracted from CPHD XML metadata

### SICD vs CPHD

| Aspect | CPHD | SICD |
|--------|------|------|
| **Domain** | K-space (phase history) | Image domain (pixels) |
| **Size** | Smaller (compressed) | Larger (complex image) |
| **Use case** | Archival, image formation, research | Display, analysis, exploitation |
| **Metadata** | PVP, waveform, geometry | Grid, projection, image statistics |
| **Processing** | Requires IFP | Ready for visualization |

### ImageFormAlgo Tag

SICD `ImageFormation/ImageFormAlgo` values:
- **`PFA`**: Polar Format Algorithm (frequency-domain)
- **`RGAZCOMP`**: Range-azimuth compression (time-domain, RDA output)
- **`OTHER`**: Non-standard (FFBP, custom algorithms)


### Output Validation

The verification step (Step 7) confirms:
- Metadata round-trip (CPHD → SICD → read)
- Image shape matches formed array
- Collection datetime preserved
- Algorithm tag correctly set

## Summary

Successfully demonstrated:
- ✅ Metadata-driven algorithm selection (SPOTLIGHT → PFA, STRIPMAP → RDA)
- ✅ Chip extraction for fast processing (1024×1024 phase history subset)
- ✅ Automated image formation from CPHD phase history
- ✅ SICD metadata construction from CPHD metadata
- ✅ NITF export with SICD DES (Data Extension Segment)
- ✅ Round-trip verification (write → read → validate)

### Key GRDL Patterns
- `get_reader('cphd', path)` for CPHD input
- `reader.read_chip(row_start, row_end, col_start, col_end)` for subset extraction
- `build_sicd_metadata(cphd_meta, geometry, image_shape, image_form_algo)` for metadata mapping
- `SICDWriter(path, metadata=sicd_meta)` for NITF output
- Single pipeline: read → select → form → construct → write

### Next Steps
- Process full collect: replace `read_chip()` with `read_full()`
- Process multiple CPHD files in batch (loop over directory)
- Override algorithm selection (force PFA for stripmap, etc.)
- Tune algorithm parameters via configuration dict
- Open formed SICD in grdk-viewer or RemoteView
- Apply orthorectification (SICD → GeoTIFF with DEM)